In [1]:
import pickle
import re

import numpy as np
from scipy.interpolate import interp1d

from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold

import pymc as pm

## Data Preparation

In [2]:
# Loading data

batDict = pickle.load(open('../data/batDict.pkl', 'rb'))
best_polGrp = np.load('../data/best_polGrp.npy')
# best_polGrp = np.load('../data/best_polGrp_km.npy')

In [3]:
cellIds = list(batDict.keys())
n_cells = len(cellIds)

In [4]:
def parseProtocol(policy):
    policy = policy.replace('-newstructure', '').strip()

    if not re.search(r'[C%]', policy, re.IGNORECASE):
        rates = [float(x) for x in policy.split('-') if x.strip()]
        return ('C', rates)  # Format C

    m = re.match(r'([\d.]+)[Cc]\(([\d.]+)%\)-([\d.]+)[Cc]?', policy)
    if m:
        return ('B', [float(m.group(1)), float(m.group(2)) / 100, float(m.group(3))])

    raise ValueError(f"Cannot parse protocol: '{policy}'")

def parse_avg_crate(charge_policy):
    fmt, vals = parseProtocol(charge_policy)
    if fmt == 'C':
        return sum(vals) * 0.20 + 1.0 * 0.20
    cc1, bp, cc2 = vals
    return cc1 * bp + cc2 * (0.80 - bp) + 1.0 * 0.20

avgCrates = np.array([parse_avg_crate(batDict[cid]['charge_policy']) for cid in cellIds])

In [5]:
policy_list      = sorted(set(batDict[c]['charge_policy'] for c in cellIds))
policy_to_idx    = {p: i for i, p in enumerate(policy_list)}
cellPolicyId     = np.array([policy_to_idx[batDict[c]['charge_policy']]
                              for c in cellIds])

final_cellGrp  = best_polGrp[cellPolicyId]

N_GROUPS = 8

In [6]:
# Feature extraction

cycle_early = 10
cycle_late = 100

def compute_F1_F2(cell):
    '''
    Feature extraction following paper
    ---
    F1: Variance of discharge \delta Q(V) between 10th and 100th cycles
    '''
    
    early = cell['cycles'][str(cycle_early)]
    late  = cell['cycles'][str(cycle_late)]
    
    V_early, Qd_early = early['V'], early['Qd']
    V_late,  Qd_late  = late['V'],  late['Qd']

    # Common voltage grid
    v_min = max(V_early.min(), V_late.min())
    v_max = min(V_early.max(), V_late.max())
    if v_min >= v_max:
        return np.nan, np.nan
    v_grid = np.linspace(v_min, v_max, 500)

    f_early = interp1d(V_early, Qd_early, bounds_error=False, fill_value=np.nan)
    f_late  = interp1d(V_late,  Qd_late,  bounds_error=False, fill_value=np.nan)
    
    delta_Q = f_late(v_grid) - f_early(v_grid)
    delta_Q = delta_Q[~np.isnan(delta_Q)]
    
    if len(delta_Q) == 0:
        return np.nan, np.nan
    
    return np.var(delta_Q), np.min(delta_Q)

In [7]:
F1_arr = np.zeros(n_cells)
F2_arr = np.zeros(n_cells)
F3_arr = np.zeros(n_cells)

for i, cid in enumerate(cellIds):
    
    cell = batDict[cid]
    
    # F3: discharge capacity at cycle 2 (index 2 because cycle 1 is index 0)
    F3_arr[i] = cell['summary']['QD'][2]   # in Ah
    
    f1, f2 = compute_F1_F2(cell)
    F1_arr[i] = f1
    F2_arr[i] = f2

In [8]:
def cell_lifetime_days(cell):
    
    cycle_durations = []
    
    for ck in cell['cycles']:
        t = cell['cycles'][ck]['t']
        if len(t) > 1:
            cycle_durations.append(t[-1] - t[0])
    
    if not cycle_durations:
        return np.nan
    
    return np.cumsum(cycle_durations)[-1] / 60 / 24

In [9]:
# mean avg c-rate per group
group_g = np.array([avgCrates[final_cellGrp == g].mean() for g in range(N_GROUPS)])

In [10]:
X = np.column_stack([F1_arr, F2_arr, F3_arr])
y_days = np.array([cell_lifetime_days(batDict[cid]) for cid in cellIds])

# Split into training and testing
# X_train, X_test, y_train, y_test = train_test_split(X, y_days, test_size=0.20, random_state=42, stratify=final_cellGrp)

In [11]:
# Apply a standard scalar

## Bayesian Hierarchical Model

In [12]:
n_features = X.shape[1]

with pm.Model() as BH_model:

    ######################################
    # Level 3: Hyperprior gamma
    ######################################
    gamma_intercept = pm.Normal('gamma_intercept', mu=0, sigma=10, shape=n_features+1)
    gamma_slope = pm.Normal('gamma_slope', mu=0, sigma=10, shape=n_features+1)

    ######################################
    # Level 2: Group prior
    ######################################

    # Random effect standard deviation
    sigma_theta = pm.Exponential('sigma_theta', 1.0, shape=n_features+1)
    
    # Non‑centered offsets 
    theta_raw = pm.Normal('theta_raw', mu=0, sigma=1, shape=(N_GROUPS, n_features+1))
    
    # Mean of theta_j
    mu_theta = gamma_intercept + gamma_slope * group_g[:, None]
    
    # Group‑specific coefficients
    theta = pm.Deterministic('theta', mu_theta + sigma_theta * theta_raw)

    ######################################
    # Level 1: Observation model
    ######################################
    sigma_y = pm.Exponential('sigma_y', 1.0)
    # theta_features = theta[final_cellGrp, 1:]  # shape (n_cells, n_features)
    # mu = theta[final_cellGrp, 0] + pm.math.sum(theta[final_cellGrp, 1:] * X_train, axis=1)
    mu = theta[final_cellGrp, 0] + pm.math.sum(theta[final_cellGrp, 1:] * X, axis=1)
    y_obs = pm.Normal('y_obs', mu=mu, sigma=sigma_y, observed=y_days)

In [13]:
with BH_model:
    trace = pm.sample(draws=1000, tune=1000, chains=2, cores=1)

Initializing NUTS using jitter+adapt_diag...
Sequential sampling (2 chains in 1 job)
NUTS: [gamma_intercept, gamma_slope, sigma_theta, theta_raw, sigma_y]


/Users/mariamelsayed/anaconda3/envs/bayes_new/lib/python3.10/site-packages/rich/live.py:260: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Sampling 2 chains for 1_000 tune and 1_000 draw iterations (2_000 + 2_000 draws total) took 38 seconds.
We recommend running at least 4 chains for robust computation of convergence diagnostics
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details


In [14]:
with BH_model:
    ppc = pm.sample_posterior_predictive(trace, random_seed=42, predictions=True)
y_pred = ppc.predictions['y_obs'].mean(dim=['chain', 'draw']).values

Sampling: [y_obs]


/Users/mariamelsayed/anaconda3/envs/bayes_new/lib/python3.10/site-packages/rich/live.py:260: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

In [15]:
rmse = np.sqrt(mean_squared_error(y_days, y_pred))
mape = mean_absolute_percentage_error(y_days, y_pred) * 100

In [16]:
print(f"RMSE: {rmse:.2f} days")
print(f"MAPE: {mape:.1f}%")

RMSE: 9.61 days
MAPE: 13.5%


## Baseline Model (Ridge Regression)